# 机组排班 csp50 —— 三方法统一报告（列生成 / Benders / LBBD）

问题与基准同各方法 notebook（50 任务、T=480、弧成本；最少 crew 数→最小总成本；
**基准：27 crew / 3139，01 直接模型证明**）。本 notebook 复用 scripts 的
crew_cg / crew_benders / crew_lbbd 运行三种方法，汇总对比并验证一致性。


In [1]:
# 环境信息
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="numpy")
import sys, platform, time
import ortools
sys.path.insert(0, "/mnt/d/exactTest/column-generation-solvers/crew_scheduling/scripts")
import crew_cg, crew_benders, crew_lbbd
print("python", platform.python_version(), "| ortools", ortools.__version__)


python 3.10.20 | ortools 9.15.6755


In [2]:
# ---- 方法一：列生成 ----
print("=" * 60)
print("方法一：列生成（覆盖 LP + CP-SAT 定价）")
print("=" * 60)
res_cg = crew_cg.run_cg(verbose=True)

# ---- 方法二：Benders ----
print()
print("=" * 60)
print("方法二：Benders（选列主问题 + 覆盖 LP 子问题）")
print("=" * 60)
res_bd = crew_benders.run_benders(verbose=True)

# ---- 方法三：LBBD（主方法）----
print()
print("=" * 60)
print("方法三：LBBD（选列主问题 + 覆盖逻辑割）")
print("=" * 60)
res_lb = crew_lbbd.run_lbbd(verbose=True)


方法一：列生成（覆盖 LP + CP-SAT 定价）
iter 1: LP=23000000.0 | CP-SAT rc=-1999808.0 | 加列 216 | 列 50


iter 2: LP=3139.0 | CP-SAT rc=0.0 | 加列 0 | 列 266
LP 下界 = 3139.0 | 整数恢复 OPTIMAL obj=3139.0 | crew 27

方法二：Benders（选列主问题 + 覆盖 LP 子问题）
iter 1: SP=3139.0 | MP=3139.0 theta=3139.0 | y选中 7 | 割 1
iter 2: SP=36000809.0 | MP=3139.0 theta=3139.0 | y选中 266 | 割 2
iter 3: SP=3139.0 | MP=3139.0 theta=3139.0 | y选中 266 | 割 3
整数修复: OPTIMAL obj=3139.0 crew=27
Benders 下界 3139.0 vs 修复 3139.0 | gap -0.0000%

方法三：LBBD（选列主问题 + 覆盖逻辑割）
iter 1: master_obj=0.0 | 选中 0 列 | 未覆盖 50 | 逻辑割 0
iter 2: master_obj=3139.0 | 选中 27 列 | 未覆盖 0 | 逻辑割 50
全部任务覆盖 -> LBBD 收敛，主问题 = 完整池覆盖 IP（证明最优）


In [3]:
# ---- 统一汇总与一致性验证 ----
def route_set(routes):
    if isinstance(routes, dict):
        routes = list(routes.values())
    return set(frozenset(r) for r in routes)

s_cg = route_set(res_cg["routes"])
s_bd = route_set(res_bd["routes"])
s_lb = route_set(res_lb["routes"])
print("三方法 crew 集合一致:", s_cg == s_bd == s_lb, "| crew 数:", len(s_cg))
print()
print("=" * 90)
print("统一结果对比（基准最优：27 crew / 3139）")
print("=" * 90)
rows = [
    ("02 列生成", res_cg["ip"], f"{res_cg['iterations']} 轮", f"LP 下界 {round(res_cg['lp'],4)}"),
    ("03 Benders", res_bd["ip"], f"{res_bd['iterations']} 轮 / {res_bd['cuts']} 割", f"下界 {round(res_bd['lb'],4)}"),
    ("05 LBBD", res_lb["obj"], f"{res_lb['iterations']} 轮 / {res_lb['cuts']} 逻辑割", "主问题=完整池 IP"),
]
print(f"{'方法':<12}{'目标':>10}{'迭代/割':>22}  证明机制")
for name, obj, itc, mech in rows:
    print(f"{name:<12}{obj:>10}{itc:>22}  {mech}")
print()
print("与基准 3139 一致:", res_cg["ip"] == res_bd["ip"] == res_lb["obj"] == 3139)
print("最优 crew 路线（三方法一致，示例前 8 条）:")
for r in sorted(s_cg, key=lambda s: -len(s))[:8]:
    print("  ", tuple(r))


三方法 crew 集合一致: True | crew 数: 27

统一结果对比（基准最优：27 crew / 3139）
方法                  目标                  迭代/割  证明机制
02 列生成          3139.0                   2 轮  LP 下界 3139.0
03 Benders      3139.0             3 轮 / 3 割  下界 3139.0
05 LBBD         3139.0          2 轮 / 50 逻辑割  主问题=完整池 IP

与基准 3139 一致: True
最优 crew 路线（三方法一致，示例前 8 条）:
   (32, 35, 21)
   (25, 18, 31)
   (16, 9, 23)
   (6, 15)
   (40, 46)
   (24, 14)
   (44, 37)
   (1, 10)


## 统一报告与结论

三种方法各自独立运行并**一致得到 27 crew / 3139**：

- **02 列生成**：2 轮收敛，LP 下界 = 3139 = 整数恢复（该实例覆盖 LP 松弛恰为整数，与 vrptw 同现象）。
- **03 Benders**：3 轮 / 3 条对偶最优性割，下界 = 整数修复 = 3139。
- **05 LBBD**：2 轮 / 50 条覆盖逻辑割，主问题等价完整池 IP ⇒ 证明最优。

最优性由 01 直接模型（K=26 不可行 + K=27 最优）与三种分解方法多重确认。
**基准最优值来源**：本家族 01_direct 自证（OR-Library csp50 文献实例，此前草稿 pool_opt 亦为 27/3139）。
